In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import tkinter as tk
from tkinter import Entry, Label, Button, Text, Scrollbar, END, font

def scrape_text(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    text = soup.get_text(separator=" ", strip=True)
    return text

def load_database(database_path):
    df = pd.read_csv(database_path)
    print(df.head()) 
    return df

def train_model(texts, labels, model_type="logistic_regression"):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(texts)
    y = labels

    # 🔄 75% train, 25% test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

    if model_type == "logistic_regression":
        model = LogisticRegression(max_iter=1000)
    elif model_type == "svm":
        model = SVC(kernel='linear')
    elif model_type == "naive_bayes":
        model = MultinomialNB()

    model.fit(X_train, y_train)

    return model, vectorizer, X_test, y_test

def find_best_match(user_text, vectorizer, df):
    user_vector = vectorizer.transform([user_text])
    X = vectorizer.transform(df['text'])
    similarities = cosine_similarity(user_vector, X)
    best_match_idx = similarities.argmax()
    return best_match_idx

def run_program(database_path):
    user_url = url_entry.get()  
    user_text = scrape_text(user_url)      
    df = load_database(database_path)  

    if 'text' not in df.columns or 'label' not in df.columns:
        result_text.config(state="normal")
        result_text.delete(1.0, END)
        result_text.insert(END, "Error: Dataset must have 'text' and 'label' columns.\n")
        result_text.config(state="disabled")
        return

    models = ["logistic_regression", "svm", "naive_bayes"]
    accuracies = {}
    model_predictions = {}

    for model_type in models:
        model, vectorizer, X_test, y_test = train_model(df['text'], df['label'], model_type=model_type)

        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)

        accuracies[model_type] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

        user_vector = vectorizer.transform([user_text])
        predicted_label = model.predict(user_vector)[0]
        model_predictions[model_type] = predicted_label

    result_text.config(state="normal")
    result_text.delete(1.0, END)
    result_text.insert(END, f"Model Evaluation Metrics on the Dataset (25% Test):\n")
    for model_type, metrics in accuracies.items():
        result_text.insert(END, f"{model_type.replace('_', ' ').title()}:\n")
        result_text.insert(END, f"  Accuracy : {metrics['accuracy']:.2f}\n")
        result_text.insert(END, f"  Precision: {metrics['precision']:.2f}\n")
        result_text.insert(END, f"  Recall   : {metrics['recall']:.2f}\n")
        result_text.insert(END, f"  F1-Score : {metrics['f1']:.2f}\n\n")

    result_text.insert(END, "\n")
    result_text.insert(END, f"Predictions for the URL (using different models):\n")
    for model_type, predicted_label in model_predictions.items():
        result_text.insert(END, f"{model_type.replace('_', ' ').title()} predicted label: {predicted_label}\n")
    result_text.insert(END, "\n")

    best_match_idx = find_best_match(user_text, vectorizer, df)
    matched_row = df.iloc[best_match_idx]

    result_text.insert(END, f"Matched Page ID: {matched_row['page_id']}\n\n")
    result_text.insert(END, f"Matched Text:\n{matched_row['text']}\n\n")
    result_text.insert(END, f"Predicted Pattern Category: {matched_row['pattern category']}\n\n")
    result_text.insert(END, f"Label (0=Benign, 1=Dark Pattern): {matched_row['label']}\n")
    result_text.config(state="disabled")

# ---------------- GUI Setup ----------------
window = tk.Tk()
window.title("DARK PATTERN DETECTOR")
window.geometry("900x700")
window.configure(bg="#E8E8E8")

default_font = font.nametofont("TkDefaultFont")
default_font.configure(size=12)
window.option_add("*Font", default_font)

url_entry_label = Label(window, text="Enter the URL to scrape:", bg="#E8E8E8", font=("Helvetica", 14))
url_entry_label.pack(pady=10)

url_entry = Entry(window, width=80, font=("Helvetica", 12))
url_entry.pack(pady=10)

# ⚠️ Update the path to match your dataset location
csv_path = r"C:\Users\chaitanya suyesha\Downloads\screenshots\dark_pattern_augmented_5000.csv"

run_button = Button(window, text="Search Patterns", command=lambda: run_program(csv_path), bg="#4CAF50", fg="white", font=("Helvetica", 12))
run_button.pack(pady=10)

result_text_label = Label(window, text="Results:", bg="#E8E8E8", font=("Helvetica", 14))
result_text_label.pack(pady=10)

result_frame = tk.Frame(window)
result_frame.pack(pady=10)

result_text = Text(result_frame, height=25, width=100, wrap="word", font=("Helvetica", 12))
result_text.pack(side="left")

scrollbar = Scrollbar(result_frame, command=result_text.yview)
scrollbar.pack(side="right", fill="y")
result_text.config(yscrollcommand=scrollbar.set)

window.mainloop()